In [4]:
import ifcopenshell
import ifcopenshell.util.element as Element
from collections import Counter

IFC_PATH = "dataset/210_King_Merged.ifc"



In [8]:
def load_model(path: str):
    model = ifcopenshell.open(path)
    print(f"Loaded: {path}")
    print(f"IFC schema: {model.schema}")
    return model


def basic_project_info(model):
    print("\n=== Project Info ===")
    project = model.by_type("IfcProject")[0]
    print(f"Project name: {project.Name}")
    print(f"Description : {project.Description}")

    # Units
    units = project.UnitsInContext
    if units:
        unit_names = [u.is_a() for u in units.Units]
        print(f"Unit types defined: {Counter(unit_names)}")


def entity_type_counts(model, top_n=25):
    print(f"\n=== Entity counts (top {top_n} by frequency) ===")
    counts = Counter(e.is_a() for e in model)
    for ifc_type, n in counts.most_common(top_n):
        print(f"{ifc_type:35s} {n}")
    print(f"\nTotal entities in file: {sum(counts.values())}")
    print(f"Total distinct IFC types: {len(counts)}")
    return counts


def spatial_structure(model, max_children_per_level=15):
    """Walk Project -> Site -> Building -> Storey -> Elements (IfcRelAggregates / IfcRelContainedInSpatialStructure)."""
    print("\n=== Spatial Structure ===")

    def print_tree(entity, depth=0):
        indent = "  " * depth
        print(f"{indent}- {entity.is_a()}: {getattr(entity, 'Name', None)}")

        # children via aggregation (e.g. Site->Building->Storey)
        if hasattr(entity, "IsDecomposedBy"):
            for rel in entity.IsDecomposedBy:
                for child in rel.RelatedObjects[:max_children_per_level]:
                    print_tree(child, depth + 1)

        # elements contained in this spatial structure element (e.g. walls in a storey)
        if entity.is_a("IfcSpatialStructureElement") and hasattr(entity, "ContainsElements"):
            for rel in entity.ContainsElements:
                elems = rel.RelatedElements
                counts = Counter(e.is_a() for e in elems)
                summary = ", ".join(f"{k}({v})" for k, v in counts.most_common())
                print(f"{indent}  contains -> {summary}")

    project = model.by_type("IfcProject")[0]
    print_tree(project)


def sample_element_properties(model, ifc_type="IfcWall", n=1):
    print(f"\n=== Sample properties for {ifc_type} ===")
    elements = model.by_type(ifc_type)
    if not elements:
        print(f"No {ifc_type} elements found in this model.")
        return

    for el in elements[:n]:
        print(f"\n{el.is_a()} | GlobalId={el.GlobalId} | Name={el.Name}")

        psets = Element.get_psets(el)
        if not psets:
            print("  (no property sets found)")
            continue
        for pset_name, props in psets.items():
            print(f"  Pset: {pset_name}")
            for prop_name, value in props.items():
                if prop_name == "id":
                    continue
                print(f"    {prop_name}: {value}")


def material_and_type_info(model, ifc_type="IfcWall", n=1):
    print(f"\n=== Type/material info for {ifc_type} ===")
    elements = model.by_type(ifc_type)
    for el in elements[:n]:
        etype = Element.get_type(el)
        print(f"{el.Name} -> Type object: {etype.Name if etype else None}")
        material = Element.get_material(el)
        print(f"Material: {material}")



In [9]:
model = load_model(IFC_PATH)
basic_project_info(model)
entity_type_counts(model)
spatial_structure(model)
sample_element_properties(model, "IfcWall", n=1)
sample_element_properties(model, "IfcDoor", n=1)

Loaded: dataset/210_King_Merged.ifc
IFC schema: IFC2X3

=== Project Info ===
Project name: Project Number
Description : None
Unit types defined: Counter({'IfcConversionBasedUnit': 4, 'IfcSIUnit': 1})

=== Entity counts (top 25 by frequency) ===
IfcPolyLoop                         579650
IfcFaceOuterBound                   579648
IfcFace                             579648
IfcCartesianPoint                   504190
IfcPropertySingleValue              88745
IfcRelDefinesByProperties           74065
IfcPropertySet                      73804
IfcAxis2Placement3D                 35379
IfcShapeRepresentation              20138
IfcPolyline                         14753
IfcLocalPlacement                   13660
IfcProductDefinitionShape           13638
IfcStyledItem                       11867
IfcExtrudedAreaSolid                10555
IfcAxis2Placement2D                 9837
IfcDistributionPort                 9824
IfcRelConnectsPortToElement         9824
IfcMappedItem                       6971

# Load CSV NODES, EDGES

In [2]:
import pandas as pd

edges_df = pd.read_csv('edges.csv')
nodes_df = pd.read_csv('nodes.csv')

print(f"Nodes: {len(nodes_df)} | {nodes_df.shape}", f"Edges: {len(edges_df)} | {edges_df.shape}")
print()
print("Edges info")
print(edges_df.info())
print()
print("Nodes info")
print(nodes_df.info())

Nodes: 413 | (413, 11) Edges: 671 | (671, 3)

Edges info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 671 entries, 0 to 670
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   source_id  671 non-null    object
 1   target_id  671 non-null    object
 2   rel_type   671 non-null    object
dtypes: object(3)
memory usage: 15.9+ KB
None

Nodes info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413 entries, 0 to 412
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           413 non-null    object 
 1   type         413 non-null    object 
 2   name         409 non-null    object 
 3   storey_id    365 non-null    object 
 4   storey_name  365 non-null    object 
 5   min_x        351 non-null    float64
 6   min_y        351 non-null    float64
 7   min_z        351 non-null    float64
 8   max_x        351 non-null    float64
 9   max_y        3

In [9]:
nodes_df.describe()

,min_x,min_y,min_z,max_x,max_y,max_z
count,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000
mean,-37.244653,-11.119950,56.193659,-25.885620,-3.106549,64.033604
std,39.485229,29.302837,2.284260,44.107594,29.520313,4.118771
min,-101.522350,-66.211290,53.750001,-89.223600,-61.965332,53.750001
25%,-70.565793,-30.092399,55.791668,-66.913902,-25.313273,62.916668
50%,-40.042432,-10.355406,55.791668,-29.074957,-4.342537,65.708335
75%,-9.798422,7.366556,55.791668,13.149877,18.240586,66.562501
max,47.827667,61.818623,67.002219,99.935848,69.878896,80.343751


In [4]:
nodes_df.isna().sum()

id              0
type            0
name            4
storey_id      48
storey_name    48
min_x          62
min_y          62
min_z          62
max_x          62
max_y          62
max_z          62
dtype: int64

# Analyze clashes

In [2]:
"""
Breaks clashes.csv (from detect_clashes.py) down by element-type pair, so
you can tell real coordination clashes apart from expected touching
elements (wall corners, curtain wall mullions, stair/railing joints, etc.)
before tuning IGNORE_TYPE_PAIRS in detect_clashes.py.

"""

import pandas as pd

CLASHES_CSV = "clashes.csv"


def type_pair(row):
    # frozenset-style sorted tuple so A-B and B-A count as the same pair
    return tuple(sorted([row["element_a_type"], row["element_b_type"]]))


df = pd.read_csv(CLASHES_CSV)
df["type_pair"] = df.apply(type_pair, axis=1)

for issue in ["CLASH", "CLEARANCE_VIOLATION"]:
    subset = df[df.issue == issue]
    print(f"\n=== {issue} ({len(subset)} total) — by type pair ===")
    counts = subset["type_pair"].value_counts()
    print(counts.to_string())

    # For CLASH: show the pairs with the largest overlap volume first,
    # since a tiny overlap volume is more likely a modeling tolerance
    # issue (e.g. a wall finish 1mm into another surface) than a real
    # coordination clash.
    if issue == "CLASH" and len(subset):
        print("\n  Top 10 by overlap volume (metric):")
        print(subset.sort_values("metric", ascending=False)
              [["element_a_type", "element_a_name", "element_b_type", "element_b_name", "metric"]]
              .head(10).to_string(index=False))


=== CLASH (34 total) — by type pair ===
type_pair
(IfcBuildingElementProxy, IfcWallStandardCase)    32
(IfcSlab, IfcSlab)                                 2

  Top 10 by overlap volume (metric):
     element_a_type                               element_a_name          element_b_type                                                  element_b_name    metric
            IfcSlab               Floor:BLDG1-FLR-Generic:339256                 IfcSlab                                  Floor:BLDG2-FLR-Generic:346979 2543.2405
IfcWallStandardCase Basic Wall:DoubleCurvedWalls-INT - 5":479628 IfcBuildingElementProxy SwingPanels_Lounge:SwingPanels_Lounge:SwingPanels_Lounge:494587   61.8506
IfcWallStandardCase Basic Wall:DoubleCurvedWalls-INT - 5":479628 IfcBuildingElementProxy SwingPanels_Lounge:SwingPanels_Lounge:SwingPanels_Lounge:494587   61.8506
IfcWallStandardCase Basic Wall:DoubleCurvedWalls-INT - 5":479638 IfcBuildingElementProxy SwingPanels_Lounge:SwingPanels_Lounge:SwingPanels_Lounge:494587 